# Validator Node — Unit Tests
Step-by-step testing of the Validator Node functions.
Run each cell independently to inspect intermediate outputs and logs.

**Pre-requisites before running:**
1. `OPENAI_API_KEY` set in `.env`
2. Qdrant running with the `openapi_reference` collection indexed (run `test_rag.ipynb` first)

In [1]:
# Step 1 — Imports and setup
# Run this cell first before any other cell.

from config import get_logger
from config.paths import TSPEC_DATA_TEST_FILE
from nodes.reader import reader_node
from nodes.planner import planner_node
from nodes.extractor import extractor_node
from nodes.reflector import reflector_node
from nodes.validator import validator_node, _validate_structurally
from schemas.rules import ValidatedRule, ValidationError
from graph.conditions import should_loop_or_build

logger = get_logger(__name__)
logger.info("Imports OK.")

/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-30 22:54:14 [INFO] __main__: Imports OK.


Using Device: cuda


In [2]:
# Step 2 — Run Reader → Planner → Extractor → Reflector to prepare state
# Limited to 2 high-priority sections to keep cost manageable.

reader_result = reader_node({
    "main_doc_path": TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
})
logger.info(f"Reader OK — {len(reader_result['parsed_sections'])} section(s).")

planner_result = planner_node({
    **reader_result,
    "main_doc_path": TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
})
logger.info(f"Planner OK — {len(planner_result['extraction_plan']['sections_to_extract'])} section(s) in plan.")

full_plan = planner_result["extraction_plan"]
high_priority = [s for s in full_plan["sections_to_extract"] if s["priority"] == "high"]
test_plan = {**full_plan, "sections_to_extract": high_priority[:2]}

extractor_result = extractor_node({
    **reader_result,
    "main_doc_path":       TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
    "extraction_plan":     test_plan,
    "validation_errors":   [],
    "iteration_count":     0,
})
logger.info(f"Extractor OK — {len(extractor_result['raw_rules'])} raw rule(s).")

reflector_result = reflector_node({
    **reader_result,
    **extractor_result,
    "main_doc_path":       TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
    "extraction_plan":     test_plan,
    "validation_errors":   [],
    "iteration_count":     1,
})
flagged = sum(1 for r in reflector_result["reflected_rules"] if r["flagged"])
logger.info(f"Reflector OK — {len(reflector_result['reflected_rules'])} rule(s), {flagged} flagged.")

2026-03-30 22:54:14 [INFO] nodes.reader: Reader Node started.
2026-03-30 22:54:14 [DEBUG] nodes.reader: Parsed 238 relevant sections from main doc.
2026-03-30 22:54:14 [DEBUG] nodes.reader: Loaded 0 auxiliary document(s).
2026-03-30 22:54:14 [INFO] tools.document_tools: discover_specs: directory mode — found 1 .md files
2026-03-30 22:54:14 [DEBUG] tools.document_tools: discover_specs: returning 1 entries
2026-03-30 22:54:14 [DEBUG] tools.document_tools: Loaded 1 OpenAPI reference file(s) from '/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/src/config/../../data/references/openapi_reference'.
2026-03-30 22:54:14 [INFO] nodes.reader: Reader Node complete.
2026-03-30 22:54:14 [INFO] __main__: Reader OK — 238 section(s).
2026-03-30 22:54:14 [INFO] nodes.planner: Planner Node started.
2026-03-30 22:54:14 [DEBUG] nodes.planner: Sending 238 section(s) to Planner LLM.
2026-03-30 22:54:14 [DEBUG] open

In [3]:
# Step 3 — Test _validate_structurally()
# Verifies that a valid rule passes and a malformed rule fails with a clear reason.

# Happy path — valid rule
valid_rule = reflector_result["reflected_rules"][0]
result_dict, error_dict = _validate_structurally(valid_rule)
assert result_dict is not None and error_dict is None
logger.info(f"_validate_structurally OK — valid rule passed.")

# Failure path — missing required field
malformed_rule = {**valid_rule}
del malformed_rule["rule_text"]  # remove required field
result_dict, error_dict = _validate_structurally(malformed_rule)
assert result_dict is None and error_dict is not None
assert error_dict["stage"] == "structural"
logger.info(f"_validate_structurally OK — malformed rule caught: {error_dict['reason'][:100]}")

2026-03-30 22:55:26 [INFO] __main__: _validate_structurally OK — valid rule passed.
2026-03-30 22:55:26 [INFO] __main__: _validate_structurally OK — malformed rule caught: 1 validation error for ValidatedRule
rule_text
  Field required [type=missing, input_value={'section


In [4]:
# Step 4 — Test validator_node() end-to-end
# Runs structural + semantic validation on all reflected rules.

validator_state = {
    **reader_result,
    **extractor_result,
    **reflector_result,
    "main_doc_path":       TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
    "extraction_plan":     test_plan,
    "iteration_count":     1,
}

validator_result = validator_node(validator_state)

assert "validated_rules" in validator_result
assert "validation_errors" in validator_result
assert isinstance(validator_result["validated_rules"], list)
assert isinstance(validator_result["validation_errors"], list)

total     = len(reflector_result["reflected_rules"])
passed    = len(validator_result["validated_rules"])
failed    = len(validator_result["validation_errors"])
error_rate = failed / total if total > 0 else 0.0

logger.info(f"validator_node OK")
logger.info(f"  total     : {total} rule(s)")
logger.info(f"  validated : {passed} passed")
logger.info(f"  errors    : {failed} failed")
logger.info(f"  error rate: {error_rate:.1%}")

2026-03-30 22:55:26 [INFO] nodes.validator: Validator Node started.
2026-03-30 22:55:26 [DEBUG] nodes.validator: Validating rule 1/4 [322]: The IS operation 'createMOI' maps to an OpenAPI path with HT
2026-03-30 22:55:26 [DEBUG] openai._base_client: Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Helper-Method': 'chat.completions.parse', 'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-17666838-ecea-4be7-b597-082f11b175cc', 'post_parser': <function Completions.parse.<locals>.parser at 0x7e9f09662700>, 'content': None, 'json_data': {'messages': [{'content': 'You are an expert in 3GPP technical specifications and OpenAPI design.\n\nYour task is to perform semantic validation on an OpenAPI rule that was automatically\nextracted from a 3GPP document and reviewed by a self-reflection step.\n\nVerify:\n  1. Is the rule_text grounded in the section content (not hallucinated)?\n  2. Is the openapi_object a v

In [5]:
# Step 5 — Test should_loop_or_build() condition
# Simulates the routing decision made by conditions.py after the Validator.

routing_state = {
    **validator_state,
    **validator_result,
}

decision = should_loop_or_build(routing_state)
assert decision in ("extractor", "builder")

logger.info(f"should_loop_or_build OK — decision: '{decision}'")
if decision == "extractor":
    logger.info(f"  → Error rate {error_rate:.1%} exceeds threshold — looping back.")
else:
    logger.info(f"  → Error rate {error_rate:.1%} within threshold — proceeding to Builder.")

2026-03-30 22:55:36 [INFO] graph.conditions: Validator result — error rate: 25.0% (1/4 rules), iteration: 1/3
2026-03-30 22:55:36 [WARNING] graph.conditions: Error rate 25.0% exceeds threshold 10%. Looping back to Extractor (iteration 2).
2026-03-30 22:55:36 [INFO] __main__: should_loop_or_build OK — decision: 'extractor'
2026-03-30 22:55:36 [INFO] __main__:   → Error rate 25.0% exceeds threshold — looping back.


In [6]:
# Step 6 — Inspect results
# Reviews validated rules and validation errors in detail.

logger.info("=== VALIDATED RULES ===")
for i, rule in enumerate(validator_result["validated_rules"]):
    ValidatedRule(**rule)  # raises ValidationError if malformed
    logger.info(f"  Rule {i + 1} [{rule['section_id']}]: {rule['rule_text'][:80]}")
    if rule.get("validation_notes"):
        logger.info(f"    notes: {rule['validation_notes'][:100]}")

logger.info("")
logger.info("=== VALIDATION ERRORS ===")
for i, err in enumerate(validator_result["validation_errors"]):
    ValidationError(**err)  # raises ValidationError if malformed
    rule = err["rule"]
    logger.info(f"  Error {i + 1} [{rule.get('section_id', '?')}] ({err['stage']}): {err['reason'][:100]}")

2026-03-30 22:55:36 [INFO] __main__: === VALIDATED RULES ===
2026-03-30 22:55:36 [INFO] __main__:   Rule 1 [322]: The IS operation 'createMOI' maps to an OpenAPI path with HTTP method PUT at the
2026-03-30 22:55:36 [INFO] __main__:     notes: The rule text accurately reflects the mapping shown in Table 12.1.1.1.1-1, where the IS operation 'c
2026-03-30 22:55:36 [INFO] __main__:   Rule 2 [322]: The IS operation 'modifyMOIAttributes' maps to OpenAPI path with HTTP methods PU
2026-03-30 22:55:36 [INFO] __main__:     notes: The rule text accurately reflects the mapping shown in Table 12.1.1.1.1-1, where the IS operation 'm
2026-03-30 22:55:36 [INFO] __main__:   Rule 3 [322]: The IS operation 'deleteMOI' maps to an OpenAPI path with HTTP method DELETE at 
2026-03-30 22:55:36 [INFO] __main__:     notes: The rule text accurately reflects the mapping of the IS operation 'deleteMOI' to the HTTP DELETE met
2026-03-30 22:55:36 [INFO] __main__: 
2026-03-30 22:55:36 [INFO] __main__: === VALIDATION 

In [ ]:
# Step 7 — Simulate loop-back: Extractor → Reflector → Validator (iteration 2)
# Verifies that on loop-back:
#   - Extractor processes ONLY the sections that had errors (not all sections).
#   - The LLM corrects only the specific failed rules.
#   - validated_rules from iteration 1 are preserved (accumulated in state).

assert decision == "extractor", (
    f"Loop-back step requires error rate > threshold. Got decision='{decision}' "
    f"(error_rate={error_rate:.1%}). Run with sections that produce errors to test loop-back."
)

errors_to_fix = validator_result["validation_errors"]
logger.info("=== LOOP-BACK ITERATION 2 ===")
logger.info(f"Feeding {len(errors_to_fix)} error(s) back to Extractor.")
logger.info(f"Sections to reprocess: {list({e['rule'].get('section_id') for e in errors_to_fix})}")

# Iteration 2 — Extractor re-runs ONLY on sections with errors
extractor_result_2 = extractor_node({
    **reader_result,
    "main_doc_path":       TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
    "extraction_plan":     test_plan,
    "validation_errors":   errors_to_fix,
    "iteration_count":     1,
})
logger.info(f"Extractor (iter 2) OK — {len(extractor_result_2['raw_rules'])} raw rule(s) "
            f"(expected ≈ {len(errors_to_fix)}, one corrected rule per error).")

# Reflector re-runs on the corrected raw_rules only
reflector_result_2 = reflector_node({
    **reader_result,
    **extractor_result_2,
    "main_doc_path":       TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
    "extraction_plan":     test_plan,
    "validation_errors":   errors_to_fix,
    "iteration_count":     2,
})
logger.info(f"Reflector (iter 2) OK — {len(reflector_result_2['reflected_rules'])} rule(s).")

# Validator re-runs on the reflected rules from iteration 2
validator_result_2 = validator_node({
    **reader_result,
    **extractor_result_2,
    **reflector_result_2,
    "main_doc_path":       TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
    "extraction_plan":     test_plan,
    "iteration_count":     2,
})

total_2     = len(reflector_result_2["reflected_rules"])
passed_2    = len(validator_result_2["validated_rules"])
failed_2    = len(validator_result_2["validation_errors"])
error_rate_2 = failed_2 / total_2 if total_2 > 0 else 0.0

# In a real graph run, validated_rules accumulates via operator.add.
# Here we simulate that manually to show the final combined result.
combined_validated = validator_result["validated_rules"] + validator_result_2["validated_rules"]

decision_2 = should_loop_or_build({
    **reader_result,
    **reflector_result_2,
    **validator_result_2,
    "iteration_count": 2,
})

logger.info(f"Validator (iter 2) OK")
logger.info(f"  rules this iter : {total_2} rule(s)")
logger.info(f"  validated       : {passed_2} passed")
logger.info(f"  errors          : {failed_2} failed")
logger.info(f"  error rate      : {error_rate_2:.1%}  (was {error_rate:.1%} in iter 1)")
logger.info(f"  decision        : '{decision_2}'")
logger.info(f"  combined total  : {len(combined_validated)} validated rule(s) accumulated")

# Show what the corrected rule looks like
logger.info("")
logger.info("=== CORRECTED RULE(S) ===")
for i, rule in enumerate(validator_result_2["validated_rules"], start=1):
    logger.info(f"Corrected rule {i} — [{rule['section_id']}]")
    logger.info(f"  rule_text      : {rule['rule_text']}")
    logger.info(f"  openapi_object : {rule['openapi_mapping']['openapi_object']}")
    logger.info(f"  openapi_field  : {rule['openapi_mapping']['openapi_field']}")
    logger.info(f"  openapi_value  : {rule['openapi_mapping']['openapi_value']}")